# Week 8: Embeddings and semantic search

An **embedding** is a list of numbers that represents what a piece of text means. Texts with similar meanings get similar numbers, even with different words, which lets you search by meaning instead of exact keywords.

**Before you start:** add the course document dataset with **Add input** (search *akada-labs-course-data*), or upload your own `.txt` file and change `DOC_PATH`. Turn **Internet on** so the embedding model can download.

Run each cell with **Shift + Enter**. Every exercise has a check cell that prints **PASS** or a hint.
At the end you get a completion code for the submission form.

In [ ]:
# Run this cell first. You don't need to read it: it holds the self-checks.
EMAIL = ""  # <- the email you applied with

import hashlib, json, os, re

_passed = set()

def ok(n):
    _passed.add(n)
    print(f"PASS: exercise {n} done.")

def hint(message):
    print(f"Not yet. Hint: {message}")

def free_text(n, text, min_sentences):
    parts = [p for p in re.split(r"[.!?]+", text or "") if len(p.split()) >= 3]
    if len(parts) >= min_sentences:
        print("Saved. A TA will read this answer.")
        ok(n)
    else:
        hint(f"write at least {min_sentences} full sentences inside the quotes.")

def completion_code(week, exercises):
    if "@" not in EMAIL:
        return print("Put your email in the first cell, run it, then run this cell again.")
    missing = sorted(set(exercises) - _passed)
    if missing:
        return print(f"Almost there. Not passed yet: {missing}. Run those exercises and checks again.")
    digest = hashlib.sha256(f"akada-labs-cohort-1|{week}|{EMAIL.strip().lower()}".encode()).hexdigest()
    print(f"Well done! Your completion code is: AK-W{week:02d}-{digest[:6].upper()}")
    print("Paste it into the lab submission form.")

def get_secret(name):
    """Read an API key from Kaggle or Colab secrets. Never paste keys into a cell."""
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    if os.environ.get(name):
        return os.environ[name]
    raise RuntimeError(
        f"No secret called {name}. On Kaggle: Add-ons > Secrets. On Colab: the key icon on the left. "
        "Add it there, switch it on for this notebook, then run this cell again.")

print("Ready.")

In [ ]:
!pip install -q sentence-transformers

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

# A small model that understands many languages, so it has a chance with Pidgin
embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

def cosine(a, b):
    a = a / np.linalg.norm(a, axis=-1, keepdims=True)
    b = b / np.linalg.norm(b, axis=-1, keepdims=True)
    return a @ b.T

print(cosine(embedder.encode(["I love jollof rice"]), embedder.encode(["Jollof is my favourite food", "The bus is late"])))

These 20 sentences come in 10 pairs that mean the same thing: sentence 0 goes with 1, 2 with 3, and so on.

In [ ]:
sentences = [
    "The price of rice has gone up again.", "Rice is more expensive than last month.",
    "NEPA took the light since morning.", "There has been no electricity all day.",
    "The traffic on Third Mainland Bridge is terrible.", "Getting across the bridge into Lagos Island takes forever.",
    "I passed my WAEC exams.", "I got good results in my secondary school final exams.",
    "The bank sent me an alert for my salary.", "My wages just landed in my account.",
    "Abeg, wetin be the time?", "Excuse me, what time is it?",
    "The Super Eagles won the match.", "Nigeria's football team got the victory.",
    "My phone battery is almost flat.", "I need to charge my phone soon.",
    "It rained heavily and the road flooded.", "The street is under water after the downpour.",
    "She is learning to code in Python.", "She is studying programming with Python.",
]

## Exercise 1: Embed and compare

Embed all 20 sentences into `vectors` with `embedder.encode(...)`, then compute the 20×20 similarity matrix `sim` with `cosine`.

In [ ]:
vectors = None
sim = None

print(None if sim is None else sim.shape)

In [ ]:
try:
    if np.shape(sim) != (20, 20):
        hint("sim = cosine(vectors, vectors) should be 20 by 20.")
    else:
        ok(1)
except Exception as error:
    hint(f"something above isn't finished yet ({type(error).__name__}: {error}). Run the cells in order.")

## Exercise 2: Find the pairs

For each sentence, find the **other** sentence most similar to it. Store the index in `best_match` (a list of 20 numbers).

Tip: copy `sim`, set the diagonal to -1 with `np.fill_diagonal`, then use `.argmax(axis=1)`.

In [ ]:
best_match = None

for i, j in enumerate([] if best_match is None else best_match):
    print(sentences[i], "->", sentences[j])

In [ ]:
try:
    right = sum(int(best_match[i]) == (i ^ 1) for i in range(20))
    if right >= 16:
        print(f"{right} of 20 sentences found their partner.")
        ok(2)
    else:
        hint(f"only {right} of 20 matched their partner. Did you set the diagonal to -1 first?")
except Exception as error:
    hint(f"something above isn't finished yet ({type(error).__name__}: {error}). Run the cells in order.")

## Search a real document

Now search a whole document: a public Nigerian text, such as the **Nigeria Data Protection Act 2023**.
We split it into chunks of about 120 words and embed each chunk.

In [ ]:
from pathlib import Path

DOC_PATH = Path("/kaggle/input/akada-labs-course-data/document.txt")
text = DOC_PATH.read_text(encoding="utf-8")

words = text.split()
chunks = [" ".join(words[i:i + 120]) for i in range(0, len(words), 100)]
print(len(words), "words ->", len(chunks), "chunks")
print(chunks[5][:300])

## Exercise 3: Semantic search

Embed every chunk into `chunk_vectors`. Then write `search(question, k=3)` that embeds the question and returns the `k` most similar chunks, best first. The check asks 3 questions you can't see.

In [ ]:
chunk_vectors = None

def search(question, k=3):
    # embed the question, compute cosine with chunk_vectors, return the top k chunks
    pass

for hit in search("What must a company do after a data breach?") or []:
    print(hit[:200], "\n")

In [ ]:
try:
    # Course team: keep these in step with the document. Each pair is a question and
    # words that should appear in the top result.
    HIDDEN = [
        ("How quickly must a data breach be reported to the regulator?", ["72 hours"]),
        ("Which body enforces this law?", ["Commission"]),
        ("When can personal data be processed without asking the person?", ["consent"]),
    ]
    found = sum(any(w.lower() in search(q, k=1)[0].lower() for w in ws) for q, ws in HIDDEN)
    if found == len(HIDDEN):
        ok(3)
    else:
        hint(f"{found} of {len(HIDDEN)} hidden questions found the right chunk. Check you sort from most to least similar.")
except Exception as error:
    hint(f"something above isn't finished yet ({type(error).__name__}: {error}). Run the cells in order.")

A simple keyword search, for comparison:

In [ ]:
def keyword_search(question, k=3):
    q = set(question.lower().split())
    scores = [len(q & set(c.lower().split())) for c in chunks]
    return [chunks[i] for i in np.argsort(scores)[::-1][:k]]

## Exercise 4: Ask in Pidgin

Write a question about the document in **Pidgin**, for example *"Wetin go happen if company leak my data?"*. Run both searches and compare the top results. In `answer`, write 2 or more sentences: which did better, and why?

In [ ]:
pidgin_question = "Wetin go happen if company leak my data?"

print("SEMANTIC:", search(pidgin_question, 1))
print("KEYWORD:", keyword_search(pidgin_question, 1))

answer = """

"""

In [ ]:
try:
    free_text(4, answer, 2)
except Exception as error:
    hint(f"something above isn't finished yet ({type(error).__name__}: {error}). Run the cells in order.")

## Finish

Run the cell below. If every exercise passed, it prints your completion code.

In [ ]:
completion_code(8, [1, 2, 3, 4])